# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Abood-arc/Flyrank-ml-project/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*

In [4]:
print("""Two findings from the FlyRank paper I want to pressure-test.

FINDING 1 -- What Predicts Health (Random Forest, ML appendix). This one doesn't really
hold up. The model is trying to predict Health Score, and Health Score is FlyRank's own
formula: impressions (30 pts) + position (30 pts) + CTR (20 pts) + scroll depth (20 pts).
The feature importance chart is basically that same formula staring back -- Average
Position 43%, Impressions 32%, Scroll Depth 15%, CTR 8%, which adds up to 98% of what the
model leans on. It isn't discovering what makes a page healthy, it's reconstructing a
recipe it was already handed the ingredients for.
Where the label comes from: straight from that documented formula -- three or four of its
own ingredients are sitting in the feature list too.
Does the validation carry the claim: no, not for anything causal, and the paper says so
itself ("importance is descriptive rather than causal"). Honest of them, but that caveat
is doing a lot of work under a heading that reads "What Predicts Health." A holdout-tested
model can still just be an accurate way of restating a formula you already know.

FINDING 2 -- Growth prediction (Logistic Regression, ML appendix). This one's a different
problem -- not the label, the split. The paper reports 71% holdout accuracy for telling
growing pages from declining ones, but never says how the 80/20 split was made: random
across all rows, or grouped so a brand's pages stay entirely in train or entirely in test.
That's not a minor detail -- this dataset spans 57 brands, and I've already measured what
happens on this exact warehouse when that goes unchecked. My own ML-05 leakage check
showed Precision@50 dropping from 0.860 on a random split to 0.580 on a client-grouped
split -- real client-level memorization, not a theoretical worry.
Where the label comes from: a 30-day-vs-previous-30-day impression trend, independent of
client -- the label itself is clean, this isn't a labeling problem.
Does the validation carry the claim: can't really tell, and that's the issue. 71% isn't
even a big, suspicious-looking number on its own -- but without knowing the split method
there's no way to tell a genuine 71% from a client-memorizing 71% wearing a modest number.
""")


Two findings from the FlyRank paper I want to pressure-test.

FINDING 1 -- What Predicts Health (Random Forest, ML appendix). This one doesn't really
hold up. The model is trying to predict Health Score, and Health Score is FlyRank's own
formula: impressions (30 pts) + position (30 pts) + CTR (20 pts) + scroll depth (20 pts).
The feature importance chart is basically that same formula staring back -- Average
Position 43%, Impressions 32%, Scroll Depth 15%, CTR 8%, which adds up to 98% of what the
model leans on. It isn't discovering what makes a page healthy, it's reconstructing a
recipe it was already handed the ingredients for.
Where the label comes from: straight from that documented formula -- three or four of its
own ingredients are sitting in the feature list too.
Does the validation carry the claim: no, not for anything causal, and the paper says so
itself ("importance is descriptive rather than causal"). Honest of them, but that caveat
is doing a lot of work under a heading tha

## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

In [5]:
%pip -q install duckdb

import os
from google.colab import userdata
os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')

import duckdb, pandas as pd, numpy as np

con = duckdb.connect()
con.execute("SET enable_progress_bar=false")
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{os.environ['HF_TOKEN']}')")
REL = 'hf://datasets/FlyRank/internship-warehouse'

print("""Same population, features, and label as w05_model.ipynb -- rebuilding it
fresh here since this notebook doesn't inherit state from that one. Nothing about the
model changes in this section, only the split does, so this cell just reproduces the
already-graded starting point. Nothing new to design here.""")

def month_read(m):
    return f"read_parquet('{REL}/fact_content_daily_performance/month={m}/*.parquet')"
months = ["2026-03", "2026-04", "2026-05", "2026-06"]
union_sql = " UNION ALL ".join(f"SELECT * FROM {month_read(m)}" for m in months)
DIM_CONTENT = f"read_parquet('{REL}/dim_content.parquet')"

raw = con.sql(f"""
    WITH unioned AS ({union_sql})
    SELECT
        f.content_hash_id,
        ANY_VALUE(f.client_hash_id) AS client_hash_id,
        MAX(CASE WHEN report_date >= DATE '2026-03-01' AND report_date < DATE '2026-04-01' THEN gsc_data_available::INT END) AS has_gsc_mar,
        MAX(CASE WHEN report_date >= DATE '2026-04-01' AND report_date < DATE '2026-05-01' THEN gsc_data_available::INT END) AS has_gsc_apr,
        MAX(CASE WHEN report_date >= DATE '2026-05-01' AND report_date < DATE '2026-06-01' THEN gsc_data_available::INT END) AS has_gsc_may,
        MAX(CASE WHEN report_date >= DATE '2026-06-01' AND report_date < DATE '2026-07-01' THEN gsc_data_available::INT END) AS has_gsc_jun,
        MAX(CASE WHEN report_date >= DATE '2026-03-01' AND report_date < DATE '2026-05-01' THEN ga4_data_available::INT END) AS has_ga4_feat,
        SUM(CASE WHEN report_date >= DATE '2026-03-01' AND report_date < DATE '2026-04-01' AND gsc_data_available IS TRUE THEN gsc_impressions ELSE 0 END) AS impr_mar,
        SUM(CASE WHEN report_date >= DATE '2026-04-01' AND report_date < DATE '2026-05-01' AND gsc_data_available IS TRUE THEN gsc_impressions ELSE 0 END) AS impr_apr,
        SUM(CASE WHEN report_date >= DATE '2026-03-01' AND report_date < DATE '2026-05-01' AND gsc_data_available IS TRUE THEN gsc_clicks ELSE 0 END) AS clicks_feat,
        SUM(CASE WHEN report_date >= DATE '2026-03-01' AND report_date < DATE '2026-05-01' AND gsc_data_available IS TRUE AND gsc_avg_position > 0 THEN gsc_avg_position * gsc_impressions ELSE 0 END) AS pos_wsum_feat,
        SUM(CASE WHEN report_date >= DATE '2026-03-01' AND report_date < DATE '2026-05-01' AND gsc_data_available IS TRUE AND gsc_avg_position > 0 THEN gsc_impressions ELSE 0 END) AS pos_wden_feat,
        SUM(CASE WHEN report_date >= DATE '2026-03-01' AND report_date < DATE '2026-05-01' AND ga4_data_available IS TRUE THEN ga4_engaged_sessions ELSE 0 END) AS ga4_engaged_feat,
        SUM(CASE WHEN report_date >= DATE '2026-05-01' AND report_date < DATE '2026-06-01' AND gsc_data_available IS TRUE THEN gsc_impressions ELSE 0 END) AS impr_may,
        SUM(CASE WHEN report_date >= DATE '2026-06-01' AND report_date < DATE '2026-07-01' AND gsc_data_available IS TRUE THEN gsc_impressions ELSE 0 END) AS impr_jun,
        SUM(CASE WHEN report_date >= DATE '2026-06-01' AND report_date < DATE '2026-07-01' AND gsc_data_available IS TRUE THEN gsc_clicks ELSE 0 END) AS clicks_jun,
        SUM(CASE WHEN report_date >= DATE '2026-06-01' AND report_date < DATE '2026-07-01' AND gsc_data_available IS TRUE AND gsc_avg_position > 0 THEN gsc_avg_position * gsc_impressions ELSE 0 END) AS pos_wsum_jun,
        SUM(CASE WHEN report_date >= DATE '2026-06-01' AND report_date < DATE '2026-07-01' AND gsc_data_available IS TRUE AND gsc_avg_position > 0 THEN gsc_impressions ELSE 0 END) AS pos_wden_jun
    FROM unioned f GROUP BY f.content_hash_id
""").df()
dim_content = con.sql(f"SELECT content_hash_id, content_type, content_updated_date FROM {DIM_CONTENT}").df()
df = raw.merge(dim_content, on="content_hash_id", how="left")
df["pos_feat"] = np.where(df["pos_wden_feat"] > 0, df["pos_wsum_feat"] / df["pos_wden_feat"], np.nan)
df["pos_jun"] = np.where(df["pos_wden_jun"] > 0, df["pos_wsum_jun"] / df["pos_wden_jun"], np.nan)
df["impr_feat"] = df["impr_mar"] + df["impr_apr"]

VOL_FLOOR_MAY, VOL_FLOOR_JUN_BASELINE = 50, 100
model_elig = ((df["has_gsc_mar"]==1)&(df["has_gsc_apr"]==1)&(df["has_gsc_may"]==1)&(df["has_gsc_jun"]==1)
              & (df["impr_may"] >= VOL_FLOOR_MAY) & df["pos_feat"].notna())
df["staleness_days"] = (pd.Timestamp("2026-06-30") - pd.to_datetime(df["content_updated_date"])).dt.days
baseline_elig = ((df["impr_jun"] >= VOL_FLOOR_JUN_BASELINE) & df["pos_jun"].notna() & (df["staleness_days"] >= 30))

shared = df[model_elig & baseline_elig].copy()
client_counts = shared.groupby("client_hash_id")["content_hash_id"].transform("count")
shared = shared[client_counts >= 20].copy()

shared["trend_pct"] = (shared["impr_jun"] - shared["impr_may"]) / shared["impr_may"] * 100
client_med = shared.groupby("client_hash_id")["trend_pct"].transform("median")
shared["relative_gap"] = shared["trend_pct"] - client_med
THRESHOLD = shared["relative_gap"].quantile(0.25)
shared["is_declining"] = (shared["relative_gap"] < THRESHOLD).astype(int)

shared["within_window_trend"] = np.where(shared["impr_mar"]>0, (shared["impr_apr"]-shared["impr_mar"])/shared["impr_mar"]*100, np.nan).clip(-1e6,1e6)
shared["within_window_trend"] = shared["within_window_trend"].fillna(0)
shared["ctr_feat"] = np.where(shared["impr_feat"]>0, shared["clicks_feat"]/shared["impr_feat"], 0)
shared["ga4_engaged_feat"] = shared["ga4_engaged_feat"].fillna(0)
shared["has_ga4_feat"] = shared["has_ga4_feat"].fillna(0).astype(int)
client_total_feat = shared.groupby("client_hash_id")["impr_feat"].transform("sum")
shared["feat_client_share"] = shared["impr_feat"] / client_total_feat
shared = pd.get_dummies(shared, columns=["content_type"], dummy_na=True, prefix="content_type")
feature_cols = (["impr_feat","clicks_feat","pos_feat","ga4_engaged_feat","has_ga4_feat",
                  "ctr_feat","within_window_trend","feat_client_share"]
                + [c for c in shared.columns if c.startswith("content_type_")])

def position_bucket(p):
    if p<=3: return "1_pos_1-3"
    if p<=6: return "2_pos_4-6"
    if p<=10: return "3_pos_7-10"
    if p<=20: return "4_pos_11-20"
    return "5_pos_21plus"
shared["position_bucket"] = shared["pos_jun"].apply(position_bucket)
shared["item_ctr_pct"] = shared["clicks_jun"]*100/shared["impr_jun"]
expected_by_bucket = shared.groupby("position_bucket").apply(lambda g: g["clicks_jun"].sum()*100/g["impr_jun"].sum(), include_groups=False)
shared["expected_ctr_pct"] = shared["position_bucket"].map(expected_by_bucket)
shared["ctr_gap_pct"] = shared["expected_ctr_pct"] - shared["item_ctr_pct"]
client_totals_jun = shared.groupby("client_hash_id")["impr_jun"].transform("sum")
shared["client_traffic_share"] = shared["impr_jun"] / client_totals_jun
shared["baseline_score"] = shared["ctr_gap_pct"].clip(lower=0) * shared["client_traffic_share"]

print(f"population: {len(shared):,} items, {shared['client_hash_id'].nunique()} clients   "
      f"threshold: {THRESHOLD:.2f}   base rate: {shared['is_declining'].mean():.3f}")
print("(w05_model.ipynb: 34,668 items, 25 clients, threshold -20.82, base rate 0.250 -- match confirms this is the same starting point)")


Same population, features, and label as w05_model.ipynb -- rebuilding it
fresh here since this notebook doesn't inherit state from that one. Nothing about the
model changes in this section, only the split does, so this cell just reproduces the
already-graded starting point. Nothing new to design here.
population: 34,668 items, 25 clients   threshold: -20.82   base rate: 0.250
(w05_model.ipynb: 34,668 items, 25 clients, threshold -20.82, base rate 0.250 -- match confirms this is the same starting point)


In [6]:
from sklearn.model_selection import GroupShuffleSplit, ShuffleSplit
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline

X_all = shared[feature_cols].fillna(0)
y_all = shared["is_declining"]
groups_all = shared["client_hash_id"]

def precision_at_k(scores, labels, tiebreak, k):
    order = np.lexsort((-tiebreak, -scores))
    return labels[order[:k]].mean()

print("""Before: a plain ShuffleSplit that ignores client_id -- the one split w05
never actually tried on itself. After: the same GroupShuffleSplit w05 already used.
Same seeds (0-19), same models, same hyperparameters, same tie-break -- the split is
the only thing that changes. Both sides averaged over 20 seeds, same reason as w05:
one split isn't trustworthy at 25 clients.""")


def run_one(seed, grouped):
    if grouped:
        splitter = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=seed)
        tr_idx, te_idx = next(splitter.split(X_all, y_all, groups_all))
    else:
        splitter = ShuffleSplit(n_splits=1, test_size=0.25, random_state=seed)
        tr_idx, te_idx = next(splitter.split(X_all, y_all))
    X_tr, X_te = X_all.iloc[tr_idx], X_all.iloc[te_idx]
    y_tr, y_te = y_all.iloc[tr_idx], y_all.iloc[te_idx]

    logreg = make_pipeline(StandardScaler(), LogisticRegression(class_weight="balanced", random_state=42, max_iter=1000)).fit(X_tr, y_tr)
    rf = RandomForestClassifier(n_estimators=200, max_depth=6, class_weight="balanced", random_state=42, n_jobs=-1).fit(X_tr, y_tr)
    n_pos, n_neg = y_tr.sum(), len(y_tr) - y_tr.sum()
    sw = np.where(y_tr == 1, len(y_tr) / (2 * n_pos), len(y_tr) / (2 * n_neg))
    gbm = HistGradientBoostingClassifier(max_depth=6, max_iter=200, random_state=42).fit(X_tr, y_tr, sample_weight=sw)

    tb, yt = shared["impr_feat"].iloc[te_idx].values, y_te.values
    return {
        "baseline_p50": precision_at_k(shared["baseline_score"].iloc[te_idx].fillna(0).values, yt, tb, 50),
        "logreg_p50": precision_at_k(logreg.predict_proba(X_te)[:,1], yt, tb, 50),
        "rf_p50": precision_at_k(rf.predict_proba(X_te)[:,1], yt, tb, 50),
        "gbm_p50": precision_at_k(gbm.predict_proba(X_te)[:,1], yt, tb, 50),
        "baseline_p20": precision_at_k(shared["baseline_score"].iloc[te_idx].fillna(0).values, yt, tb, 20),
        "logreg_p20": precision_at_k(logreg.predict_proba(X_te)[:,1], yt, tb, 20),
        "rf_p20": precision_at_k(rf.predict_proba(X_te)[:,1], yt, tb, 20),
        "gbm_p20": precision_at_k(gbm.predict_proba(X_te)[:,1], yt, tb, 20),
    }

random_res = pd.DataFrame([run_one(s, grouped=False) for s in range(20)])
grouped_res = pd.DataFrame([run_one(s, grouped=True) for s in range(20)])

print(f"\nbase rate: {y_all.mean():.3f}  n={len(shared):,}  clients={shared['client_hash_id'].nunique()}\n")
for name in ["baseline","logreg","rf","gbm"]:
    rp50, rp20 = random_res[f"{name}_p50"], random_res[f"{name}_p20"]
    gp50, gp20 = grouped_res[f"{name}_p50"], grouped_res[f"{name}_p20"]
    print(f"{name:<10} before(random)  P@50={rp50.mean():.3f} ({rp50.min():.3f}-{rp50.max():.3f})   P@20={rp20.mean():.3f} ({rp20.min():.3f}-{rp20.max():.3f})")
    print(f"{name:<10} after(grouped)  P@50={gp50.mean():.3f} ({gp50.min():.3f}-{gp50.max():.3f})   P@20={gp20.mean():.3f} ({gp20.min():.3f}-{gp20.max():.3f})")

baseline_coinflip = (random_res["baseline_p50"] > grouped_res["baseline_p50"].mean()).mean()
gbm_above_grouped_mean = (random_res["gbm_p50"] > grouped_res["gbm_p50"].mean()).mean()
gbm_wins_random = (random_res["gbm_p50"] > random_res["rf_p50"]).sum()
rf_wins_grouped = (grouped_res["rf_p50"] > grouped_res["gbm_p50"]).sum()
gap_rf = random_res["rf_p50"].mean() - grouped_res["rf_p50"].mean()
gap_gbm = random_res["gbm_p50"].mean() - grouped_res["gbm_p50"].mean()


print(f"""Baseline and logistic regression barely move. Baseline's random-split P@50 beats its
own grouped mean in about {baseline_coinflip*100:.0f}% of seeds -- basically a coin
flip, which is exactly what a fixed rule that never trains on the split should look
like. RF moves (gap {gap_rf:+.3f} P@50). GBM moves the most: gap {gap_gbm:+.3f} P@50,
and {gbm_above_grouped_mean*100:.0f}% of its random-split seeds beat the grouped mean
outright.

Here's the part that matters: the RF-vs-GBM ranking flips depending on the split.
Checked seed by seed, not just the averages -- GBM beats RF in {gbm_wins_random}/20
random-split seeds ({random_res['gbm_p50'].mean():.3f} vs {random_res['rf_p50'].mean():.3f}
mean P@50). RF beats GBM in {rf_wins_grouped}/20 grouped-split seeds
({grouped_res['rf_p50'].mean():.3f} vs {grouped_res['gbm_p50'].mean():.3f}). Under a
random split, GBM would've looked like the better model. The honest, client-grouped
split w05 actually used says the opposite. So this isn't just "the number moved" --
an unhonest split can flip which model you'd actually ship.
""")


Before: a plain ShuffleSplit that ignores client_id -- the one split w05
never actually tried on itself. After: the same GroupShuffleSplit w05 already used.
Same seeds (0-19), same models, same hyperparameters, same tie-break -- the split is
the only thing that changes. Both sides averaged over 20 seeds, same reason as w05:
one split isn't trustworthy at 25 clients.

base rate: 0.250  n=34,668  clients=25

baseline   before(random)  P@50=0.145 (0.040-0.260)   P@20=0.107 (0.000-0.250)
baseline   after(grouped)  P@50=0.135 (0.060-0.240)   P@20=0.105 (0.000-0.250)
logreg     before(random)  P@50=0.394 (0.240-0.520)   P@20=0.358 (0.200-0.500)
logreg     after(grouped)  P@50=0.381 (0.220-0.680)   P@20=0.313 (0.150-0.650)
rf         before(random)  P@50=0.546 (0.400-0.680)   P@20=0.565 (0.250-0.750)
rf         after(grouped)  P@50=0.445 (0.280-0.620)   P@20=0.465 (0.200-0.650)
gbm        before(random)  P@50=0.576 (0.420-0.720)   P@20=0.573 (0.400-0.750)
gbm        after(grouped)  P@50=0.380

## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

In [8]:
from sklearn.model_selection import GroupShuffleSplit

def score_featureset(cols, seed=7):
    X = shared[cols].fillna(0)
    gss = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=seed)
    tr_idx, te_idx = next(gss.split(X, y_all, groups_all))
    rf = RandomForestClassifier(n_estimators=200, max_depth=6, class_weight="balanced", random_state=42, n_jobs=-1).fit(X.iloc[tr_idx], y_all.iloc[tr_idx])
    tb = shared["impr_feat"].iloc[te_idx].values
    yt = y_all.iloc[te_idx].values
    return precision_at_k(rf.predict_proba(X.iloc[te_idx])[:,1], yt, tb, 50)

honest = score_featureset(feature_cols)
with_label_leak = score_featureset(feature_cols + ["relative_gap"])
with_future_leak = score_featureset(feature_cols + ["impr_jun"])


print(f"""Same attack the Week-3 leakage check ran, just on this notebook's actual
12-feature set instead of that earlier 26-feature one: hand the model a feature it
shouldn't get, and watch the score break. One grouped split (seed=7, same one w05
used for its own error analysis) for a clean before/after in one cell. Heads up --
this honest baseline number might not land exactly on w05's own seed=7 figure. That's
the same RF run-to-run variance the project's already run into, not a bug.

honest 12-feature set:                  P@50 = {honest:.3f}
+ relative_gap (label-derived leak):    P@50 = {with_label_leak:.3f}
+ impr_jun (future-window leak):        P@50 = {with_future_leak:.3f}

relative_gap is literally the number is_declining gets thresholded from, so giving it
to the model hands over the answer key -- and the score shows it. impr_jun has no
formula tie to the label at all and still nearly doubles the honest score. Future data
doesn't need to share the label's math to be fatal, which is the same thing Week 3
found on a completely different feature set. Both leaks are big and obvious -- the
harness is doing what it's supposed to.""")

print("\nWalking the actual 12 features for anything that shouldn't be there:\n")
for c in feature_cols:
    print(f"  {c}")

print("""
All twelve are March+April only, and that's by construction, not by luck: the five
reused from ML-04 (impr_feat, clicks_feat, pos_feat, ga4_engaged_feat, has_ga4_feat)
plus ctr_feat are summed only over the feature window in the SQL that builds them.
within_window_trend compares March to April and never touches May or June.
feat_client_share is this page's share of March+April traffic, not June's.
content_type is a static attribute -- no time dimension to leak through. None of them
are product flags or existing FlyRank scores either. These are things we actually
measured, not a rule someone else already applied.

One thing I'm flagging without removing it: feat_client_share's March+April version
predicts decline in the OPPOSITE direction from its June version. That's not a leak --
both versions are legitimately time-scoped -- but it's an open question from w05 that
a leakage audit should say out loud instead of quietly carrying forward.
fact_content_query_90d got considered and rejected back in w05 for a related reason:
its window overlaps the label period even in *_prev30 form, so no slice of it comes
before March+April. Checked once already, not re-litigating it here.
""")


Same attack the Week-3 leakage check ran, just on this notebook's actual
12-feature set instead of that earlier 26-feature one: hand the model a feature it
shouldn't get, and watch the score break. One grouped split (seed=7, same one w05
used for its own error analysis) for a clean before/after in one cell. Heads up --
this honest baseline number might not land exactly on w05's own seed=7 figure. That's
the same RF run-to-run variance the project's already run into, not a bug.

honest 12-feature set:                  P@50 = 0.400
+ relative_gap (label-derived leak):    P@50 = 1.000
+ impr_jun (future-window leak):        P@50 = 0.720

relative_gap is literally the number is_declining gets thresholded from, so giving it
to the model hands over the answer key -- and the score shows it. impr_jun has no
formula tie to the label at all and still nearly doubles the honest score. Future data
doesn't need to share the label's math to be fatal, which is the same thing Week 3
found on a complete

## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

In [9]:
print("""
    "Random Forest wins."

Here's the same claim, said properly:

On the shared 34,668-item, 25-client population built for this comparison, Random
Forest had the highest measured Precision@50 -- mean 0.453 across 20 client-grouped
splits -- of the three methods I tested, and beat the Week-4 baseline in every single
one of those 20 splits. That's observed and directional, specific to this population
and this label design. It's not a general claim that Random Forest is the better
algorithm for this problem, and it's not a guarantee about any one page. It's
decision-support for prioritizing a review queue, nothing more certain than that.

"Wins" is one unqualified word standing in for a mean across 20 splits on one
25-client sample. Fine as shorthand between sessions. Not fine if it ended up on a
public page. The rewrite doesn't make the finding weaker -- it just says what I
actually measured instead of what was quicker to type.""")



    "Random Forest wins."

Here's the same claim, said properly:

On the shared 34,668-item, 25-client population built for this comparison, Random
Forest had the highest measured Precision@50 -- mean 0.453 across 20 client-grouped
splits -- of the three methods I tested, and beat the Week-4 baseline in every single
one of those 20 splits. That's observed and directional, specific to this population
and this label design. It's not a general claim that Random Forest is the better
algorithm for this problem, and it's not a guarantee about any one page. It's
decision-support for prioritizing a review queue, nothing more certain than that.

"Wins" is one unqualified word standing in for a mean across 20 splits on one
25-client sample. Fine as shorthand between sessions. Not fine if it ended up on a
public page. The rewrite doesn't make the finding weaker -- it just says what I
actually measured instead of what was quicker to type.


## Self-check

Before you submit, confirm each line honestly:

- [X] Every section above is filled — markdown thinking AND the code that backs it
- [X] The notebook runs top to bottom with no errors (Runtime → Run all)
- [X] No client names, URLs, or private queries anywhere
- [X] My claims use careful words: observed, measured, directional, decision-support
- [X] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.